# Task 1: GPT-style Character-Level LLM on TinyStories

This notebook trains a small GPT model from scratch. The config name is read from the `CONFIG` environment variable (`local` or `gpu`).

## 1. Setup

In [ ]:
import os
import random
import sys
import time
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

sys.path.append("src")
from data import ROOT, get_data, load_config
from model import build_model, count_parameters
from utils import (Logger, get_device, load_checkpoint, make_scheduler,
                   peak_memory_mb, save_checkpoint)

config_name = os.environ.get("CONFIG", "local")
cfg = load_config(config_name)

random.seed(cfg["seed"])
np.random.seed(cfg["seed"])
torch.manual_seed(cfg["seed"])
torch.cuda.manual_seed_all(cfg["seed"])

device, device_name = get_device()

out_dir = ROOT / cfg["output_dir"]
logger = Logger(out_dir / "train_log.txt")
logger.log(f"config name: {config_name}")
logger.log(f"device: {device_name}")
logger.log(f"torch version: {torch.__version__}")
logger.log("config:\n" + yaml.dump(cfg, sort_keys=False))

## 2. Load Data

In [ ]:
train_loader, val_loader, char_to_idx, idx_to_char = get_data(cfg)
vocab_size = len(char_to_idx)
logger.log(f"vocab size: {vocab_size}")
logger.log(f"train sequences: {len(train_loader.dataset)}, val sequences: {len(val_loader.dataset)}")

## 3. Build Model

In [ ]:
model = build_model(cfg, vocab_size).to(device)
logger.log(f"parameters: {count_parameters(model)}")

## 4. Train

### Training setup

In [ ]:
tcfg = cfg["train"]
optimizer = torch.optim.AdamW(model.parameters(), lr=tcfg["lr"], weight_decay=tcfg["weight_decay"])
total_steps = tcfg["epochs"] * len(train_loader)
scheduler = make_scheduler(optimizer, tcfg["warmup_steps"], total_steps)
use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()

ckpt_dir = out_dir / "checkpoints"
last_path = ckpt_dir / "last.pt"
best_path = ckpt_dir / "best.pt"
history_path = out_dir / "history.csv"

start_epoch, step, best_val = 0, 0, float("inf")
history = []
if last_path.exists():
    start_epoch, step, best_val = load_checkpoint(last_path, model, optimizer, scheduler, device)
    history = pd.read_csv(history_path).iloc[:start_epoch].to_dict("records")
    logger.log(f"resumed from last.pt at epoch {start_epoch}, step {step}")
logger.log(f"bf16 autocast: {use_bf16}")


def autocast():
    if use_bf16:
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return nullcontext()


@torch.no_grad()
def evaluate():
    model.eval()
    total, count = 0.0, 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        with autocast():
            _, loss = model(x, y)
        total += loss.item() * x.size(0)
        count += x.size(0)
    model.train()
    return total / count

### Training loop

In [ ]:
grad_avg = None

for epoch in range(start_epoch, tcfg["epochs"]):
    torch.manual_seed(cfg["seed"] + epoch)
    model.train()
    epoch_start = time.time()
    interval_start = time.time()
    interval_loss, interval_n, interval_tokens = 0.0, 0, 0
    epoch_loss, epoch_n = 0.0, 0
    nan_count, spike_count = 0, 0
    grad_sum, grad_max, epoch_tokens = 0.0, 0.0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            _, loss = model(x, y)
        if not torch.isfinite(loss):
            nan_count += 1
            continue
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg["grad_clip"]).item()
        if grad_avg is not None and grad_norm > tcfg["spike_factor"] * grad_avg:
            spike_count += 1
        grad_avg = grad_norm if grad_avg is None else 0.9 * grad_avg + 0.1 * grad_norm
        grad_sum += grad_norm
        grad_max = max(grad_max, grad_norm)
        epoch_tokens += x.numel()
        optimizer.step()
        scheduler.step()
        step += 1

        interval_loss += loss.item()
        interval_n += 1
        interval_tokens += x.numel()
        epoch_loss += loss.item()
        epoch_n += 1

        if step % tcfg["log_every"] == 0:
            speed = interval_tokens / (time.time() - interval_start)
            logger.log(f"step {step} | train loss {interval_loss / interval_n:.4f} | "
                       f"lr {scheduler.get_last_lr()[0]:.6f} | grad norm {grad_norm:.3f} | "
                       f"tokens/sec {speed:.0f}")
            interval_start = time.time()
            interval_loss, interval_n, interval_tokens = 0.0, 0, 0

    train_loss = epoch_loss / epoch_n
    train_time = time.time() - epoch_start
    val_loss = evaluate()
    epoch_time = time.time() - epoch_start
    mem = peak_memory_mb(device)
    mem_text = f"{mem:.0f} MB" if mem is not None else "n/a"
    logger.log(f"epoch {epoch + 1}/{tcfg['epochs']} | train loss {train_loss:.4f} | "
               f"val loss {val_loss:.4f} | time {epoch_time:.1f}s | nan losses {nan_count} | "
               f"grad spikes {spike_count} | peak memory {mem_text}")

    history.append({"epoch": epoch + 1, "step": step, "train_loss": train_loss,
                    "val_loss": val_loss, "epoch_time": epoch_time,
                    "nan_losses": nan_count, "grad_spikes": spike_count,
                    "grad_norm_mean": grad_sum / epoch_n, "grad_norm_max": grad_max,
                    "tokens_per_sec": epoch_tokens / train_time,
                    "peak_memory_mb": mem if mem is not None else float("nan")})
    pd.DataFrame(history).to_csv(history_path, index=False)

    if val_loss < best_val:
        best_val = val_loss
        save_checkpoint(best_path, model, optimizer, scheduler, epoch + 1, step, best_val, cfg)
    save_checkpoint(last_path, model, optimizer, scheduler, epoch + 1, step, best_val, cfg)

logger.log(f"training finished, best val loss {best_val:.4f}")

## 5. Loss Curves

In [ ]:
history_df = pd.read_csv(history_path)

plt.figure(figsize=(6, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Train vs Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(out_dir / "loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()